# Data Prep: Download & Cache Prices (FMP)

This notebook downloads historical prices from Financial Modeling Prep (FMP)
based on the portfolio CSVs in `notebooks/`, then caches the combined price
matrix under `data/cache/` for reuse.


## Prerequisites
- Set your FMP API key: `export FMP_API_KEY=...`
- Install dependencies: `pip install pandas numpy requests`


In [1]:
import os
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd
import requests

FMP_API_KEY = os.getenv("FMP_API_KEY", "***REMOVED***").strip()
if not FMP_API_KEY:
    raise RuntimeError("Set FMP_API_KEY in your environment before running this notebook.")


## Load portfolios and build ticker universe


In [2]:
from pathlib import Path

PORTFOLIO_DIR = Path("notebooks")
if not PORTFOLIO_DIR.exists():
    PORTFOLIO_DIR = Path(".")

PORTFOLIO_FILES = [
    str(PORTFOLIO_DIR / f"portfolio_{i}.csv") for i in range(1, 6)
]

def load_portfolio(path):
    df = pd.read_csv(path)
    if set(df.columns) != {"ticker", "weight"}:
        raise ValueError(f"Expected columns ticker,weight in {path}")
    total = df["weight"].sum()
    if abs(total - 1.0) > 1e-6:
        raise ValueError(f"Weights in {path} must sum to 1.0 (got {total})")
    return df

portfolios = {path: load_portfolio(path) for path in PORTFOLIO_FILES}
tickers = sorted({t for df in portfolios.values() for t in df["ticker"].tolist()})
tickers


['AAPL',
 'ABBV',
 'AMZN',
 'AVGO',
 'BRK.B',
 'COST',
 'GOOGL',
 'HD',
 'JNJ',
 'JPM',
 'LLY',
 'MA',
 'META',
 'MSFT',
 'NVDA',
 'PG',
 'TSLA',
 'UNH',
 'V',
 'XOM']

## Download and cache prices


In [3]:
from pathlib import Path

END_DATE = datetime.utcnow().date()
START_DATE = END_DATE - timedelta(days=365 * 2)

cache_dir = Path("data/cache")
cache_dir.mkdir(parents=True, exist_ok=True)
combined_path = cache_dir / "fmp_prices.csv"

def _fmp_ticker_candidates(ticker):
    candidates = [ticker]
    if "." in ticker:
        candidates.append(ticker.replace(".", "-"))
    return candidates

def fetch_prices_fmp(ticker, start_date, end_date, api_key):
    last_error = None
    for candidate in _fmp_ticker_candidates(ticker):
        url = f"https://financialmodelingprep.com/api/v3/historical-price-full/{candidate}"
        params = {
            "from": start_date.strftime("%Y-%m-%d"),
            "to": end_date.strftime("%Y-%m-%d"),
            "serietype": "line",
            "apikey": api_key,
        }
        resp = requests.get(url, params=params, timeout=30)
        resp.raise_for_status()
        data = resp.json()
        historical = data.get("historical")
        if not historical:
            last_error = f"No historical data for {candidate}: {data}"
            continue
        df = pd.DataFrame(historical)
        df["date"] = pd.to_datetime(df["date"])
        df = df.set_index("date")["close"].sort_index()
        df.name = ticker
        return df
    raise ValueError(last_error or f"No historical data for {ticker}")

price_series = []
for ticker in tickers:
    ticker_path = cache_dir / f"{ticker}.csv"
    if ticker_path.exists():
        series = pd.read_csv(ticker_path, parse_dates=["date"]).set_index("date")["close"]
        series.name = ticker
    else:
        series = fetch_prices_fmp(ticker, START_DATE, END_DATE, FMP_API_KEY)
        series.to_frame("close").to_csv(ticker_path, index=True)
    price_series.append(series)

prices = pd.concat(price_series, axis=1).dropna()
prices.to_csv(combined_path, index=True)
print(f"Saved combined prices to {combined_path}")


/tmp/ipykernel_21592/406217319.py:3: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  END_DATE = datetime.utcnow().date()


Saved combined prices to data/cache/fmp_prices.csv
